# **08. Exception Handling with Functions and Classes**

### **8.1 Exception Handling Inside Functions**

**1. Handling Errors Inside a Function**

- ***A function can use `try`-`except` to handle errors itself.***

- ***It can return a safe value, such as `None`, or a friendly message when something goes wrong.***

In [1]:
def safe_get(data, key):
    try:
        return data[key]
    except KeyError:
        return f"Key '{key}' not found"

student = {"name": "Asha", "age": 22}
print(safe_get(student, "name"))
print(safe_get(student, "grade"))

Asha
Key 'grade' not found


In [2]:
def divide_numbers(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("Caught an error: Cannot divide by zero!")
        return None                    # Return None to indicate failure
    else:
        return result

print(divide_numbers(10, 2))
print(divide_numbers(10, 0))

5.0
Caught an error: Cannot divide by zero!
None


### **8.2 Exception Propagation**

**1. What is Exception Propagation?**

- ***If an exception is not handled inside a function, it moves up to the function that called it.***

- ***This continues up the chain of calls until a matching handler is found.***

- ***If no handler is found, the program terminates and prints a traceback.***

In [3]:
import traceback

def level3():
    return 10 / 0                     # error occurs here

def level2():
    return level3()                   # no handler

def level1():
    return level2()                   # no handler

try:
    level1()                          # handled here
except ZeroDivisionError as e:
    print("Caught in the caller:", e)
    print("Path:", [step.name for step in traceback.extract_tb(e.__traceback__)])

Caught in the caller: division by zero
Path: ['<module>', 'level1', 'level2', 'level3']


**2. Handling in the Middle Stops Propagation**

In [4]:
def level2_safe():
    try:
        return level3()
    except ZeroDivisionError:
        print("level2_safe handled the error")
        return 0

print(level2_safe())

level2_safe handled the error
0


### **8.3 Handling Exceptions in Function Calls**

**1. Handling at the Call Site**

- ***A function can simply raise the exception, and the code that calls the function decides how to handle it.***

- ***This keeps the function simple and lets different callers react differently.***

In [5]:
def check_age(age):
    if age < 0:
        raise ValueError(f'Age cannot be negative: {age}')
    if age > 150:
        raise ValueError(f'Age seems unrealistic: {age}')
    return f'Age {age} is valid'

for value in [25, -5, 200]:
    try:
        print(check_age(value))
    except ValueError as e:
        print("Error:", e)

Age 25 is valid
Error: Age cannot be negative: -5
Error: Age seems unrealistic: 200


### **8.4 Exception Handling in Classes**

**1. Exceptions in Methods**

- ***Methods of a class can raise exceptions when an operation is not valid.***

- ***The code that uses the object handles them with `try`-`except`.***

In [6]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        self.__balance = balance                       # private attribute

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Deposit must be positive")
        self.__balance += amount

    def withdraw(self, amount):
        if amount > self.__balance:
            raise ValueError("Insufficient balance")
        self.__balance -= amount

    @property
    def balance(self):                                 # read-only access
        return self.__balance

acc = BankAccount("Asha", 5000)

for action, amount in [("deposit", 1500), ("deposit", -10), ("withdraw", 9000)]:
    try:
        getattr(acc, action)(amount)
        print(action, amount, "-> balance:", acc.balance)
    except ValueError as e:
        print(action, amount, "-> Error:", e)

deposit 1500 -> balance: 6500
deposit -10 -> Error: Deposit must be positive
withdraw 9000 -> Error: Insufficient balance


### **8.5 Exceptions in Constructors (`__init__`)**

**1. Validating Data in `__init__`**

- ***A constructor can raise an exception if the given data is invalid.***

- ***If `__init__` raises an exception, the object is not created.***

In [7]:
class Student:
    def __init__(self, name, marks):
        if not name:
            raise ValueError("Name cannot be empty")
        if not 0 <= marks <= 100:
            raise ValueError("Marks must be between 0 and 100")
        self.name = name
        self.marks = marks

for name, marks in [("Asha", 85), ("", 70), ("Ravi", 150)]:
    try:
        s = Student(name, marks)
    except ValueError as e:
        print("Could not create student:", e)
    else:
        print("Created:", s.name, s.marks)

Created: Asha 85
Could not create student: Name cannot be empty
Could not create student: Marks must be between 0 and 100


### **8.6 Creating and Using Exception Classes in OOP**

**1. Custom Exceptions with Classes**

- ***A class can use its own custom exception to report problems.***

- ***The exception can carry the values that caused the problem.***

In [8]:
class InsufficientBalanceError(Exception):
    def __init__(self, balance, amount):
        super().__init__(f"Balance {balance} is less than requested {amount}")
        self.balance = balance
        self.amount = amount

class Account:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientBalanceError(self.balance, amount)
        self.balance -= amount
        return self.balance

account = Account(5000)

try:
    print(account.withdraw(2000))
    print(account.withdraw(8000))
except InsufficientBalanceError as e:
    print("Error:", e)
    print("Short by:", e.amount - e.balance)

3000
Error: Balance 3000 is less than requested 8000
Short by: 5000
